Developing a REST API to serve a trained AI/ML model, allowing external applications to send input data and receive predictions via HTTP requests.<br>
The API should handle requests and return responses in JSON format.


In [22]:
# ============================================================
# STEP 1: IMPORT REQUIRED LIBRARIES
# ============================================================

#core libraries
import os
import json
import joblib
import logging
import warnings
import pandas as pd
import numpy as np
from datetime import datetime

#machine learning libraries
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

#remove warnings
warnings.filterwarnings("ignore")

In [23]:
# ============================================================
# STEP 2: LOAD BREAST CANCER DATASET
# ============================================================

# Load the breast cancer dataset using scikit-learn's built-in function
df = load_breast_cancer()

# Convert the dataset into a pandas DataFrame for easier manipulation
X = pd.DataFrame(df.data, columns=df.feature_names)
y = pd.Series(df.target)

# Create a mapping of target labels to their corresponding class names
label_mapping = {
    int(index): label_name
    for index, label_name in enumerate(df.target_names)
}

print("Dataset loaded successfully")
print("Shape:", X.shape)
print("Labels:", label_mapping)

Dataset loaded successfully
Shape: (569, 30)
Labels: {0: 'malignant', 1: 'benign'}


In [24]:
# ============================================================
# STEP 3: SPLIT DATA INTO TRAINING AND TESTING SETS
# ============================================================

#train_test_split with stratification to maintain class distribution
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Training samples: 455
Testing samples: 114


In [25]:
# ============================================================
# STEP 4: SCALE FEATURES
# ============================================================

# Initialize the StandardScaler
scaler = StandardScaler()

# Fit the scaler on the training data and transform both training and testing data
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Feature scaling completed successfully")

Feature scaling completed successfully


In [26]:
# ============================================================
# STEP 5: TRAIN SMALL MODEL
# ============================================================

# Initialize a simple Logistic Regression model with increased max_iter to ensure convergence
small_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

small_model.fit(X_train_scaled, y_train)

small_predictions = small_model.predict(X_test_scaled)
small_accuracy = accuracy_score(y_test, small_predictions)

print("Small model trained successfully")
print("Small model accuracy:", round(small_accuracy, 4))

Small model trained successfully
Small model accuracy: 0.9825


In [27]:
# ============================================================
# STEP 6: TRAIN LARGE MODEL
# ============================================================

# Initialize a more complex Random Forest Classifier with increased n_estimators for better performance
large_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    random_state=42
)

large_model.fit(X_train_scaled, y_train)

large_predictions = large_model.predict(X_test_scaled)
large_accuracy = accuracy_score(y_test, large_predictions)

print("Large model trained successfully")
print("Large model accuracy:", round(large_accuracy, 4))

Large model trained successfully
Large model accuracy: 0.9561


In [28]:
# ============================================================
# STEP 7: EVALUATE BOTH MODELS
# ============================================================

# Generate classification reports for both models
print("Small Model Classification Report")
print(classification_report(y_test, small_predictions, target_names=df.target_names))

# Generate classification report for the large model
print("Large Model Classification Report")
print(classification_report(y_test, large_predictions, target_names=df.target_names))

Small Model Classification Report
              precision    recall  f1-score   support

   malignant       0.98      0.98      0.98        42
      benign       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114

Large Model Classification Report
              precision    recall  f1-score   support

   malignant       0.95      0.93      0.94        42
      benign       0.96      0.97      0.97        72

    accuracy                           0.96       114
   macro avg       0.96      0.95      0.95       114
weighted avg       0.96      0.96      0.96       114



In [29]:
# ============================================================
# STEP 8: SAVE MODELS AND SUPPORT FILES
# ============================================================

# Create a metadata dictionary to store information about the models and dataset
metadata = {
    "project_name": "REST API for AI Model using FastAPI",
    "dataset": "Breast Cancer Dataset",
    "small_model": "Logistic Regression",
    "large_model": "Random Forest Classifier",
    "small_model_accuracy": round(float(small_accuracy), 4),
    "large_model_accuracy": round(float(large_accuracy), 4),
    "features": list(X.columns),
    "number_of_features": X.shape[1],
    "classes": label_mapping,
    "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
}

# Save the models, scaler, metadata, and label mapping using joblib
joblib.dump(small_model,"models/small_model.pkl")
joblib.dump(large_model, "models/large_model.pkl")
joblib.dump(scaler, "models/scaler.pkl")
joblib.dump(metadata, "models/model_metadata.pkl")
joblib.dump(label_mapping, "models/label_mapping.pkl")

print("All models and support files saved successfully")

All models and support files saved successfully


In [30]:
# ============================================================
# STEP 9: CREATE SAMPLE API REQUEST FILE
# ============================================================

# Create a directory for test requests if it doesn't exist
sample_request = {
    "model_type": "small",
    "features": [5.1, 3.5, 1.4, 0.2]
}

# Create a batch request sample with multiple records
sample_batch_request = {
    "model_type": "large",
    "records": [
        {"features": [5.1, 3.5, 1.4, 0.2]},
        {"features": [6.2, 3.4, 5.4, 2.3]}
    ]
}

# Save the sample requests as JSON files in the test_requests directory
with open(os.path.join("test_requests", "sample_request.json"), "w") as file:
    json.dump(sample_request, file, indent=4)

# Save the batch request sample as a JSON file
with open(os.path.join("test_requests", "sample_batch_request.json"), "w") as file:
    json.dump(sample_batch_request, file, indent=4)

print("Sample request files created successfully")

Sample request files created successfully
